# Kaggle HG/MG Classification Baseline

Validate scalar processed metrics as HG/MG detection features. The notebook
selects the most completely processed Kaggle scorer by balanced HG/MG coverage
unless `MODEL_OVERRIDE` is set. MG is the positive class (`1`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display
import pandas as pd
import seaborn as sns

from notebooks.utils.classification_utils import (
    load_baseline_cohort,
    plot_confusion_matrices,
    plot_score_curves,
    run_baseline_experiments,
    split_cohort,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

In [ ]:
DATASET = "kaggle_train"
MODEL_OVERRIDE = None
SEED = 42
TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20

In [ ]:
cohort, feature_columns, selected_model, coverage = load_baseline_cohort(
    PROCESSED_ROOT,
    dataset=DATASET,
    model=MODEL_OVERRIDE,
)

print(f"Dataset: {DATASET}")
print(f"Selected scorer model: {selected_model}")
print(f"Scalar features: {feature_columns}")
display(coverage)
display(cohort.groupby("split").size().rename("retained_rows").reset_index())

In [ ]:
partitions = split_cohort(
    cohort,
    seed=SEED,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
)
display(
    pd.concat(
        {
            name: partition.groupby("split").size().rename("rows")
            for name, partition in partitions.items()
        },
        names=["partition", "split"],
    ).reset_index()
)

In [ ]:
test_results, test_predictions = run_baseline_experiments(
    partitions,
    feature_columns,
    seed=SEED,
)
display(
    test_results.sort_values("macro_f1", ascending=False).reset_index(drop=True)
)

# Test results are exploratory baseline comparisons. The test split did not
select feature columns, score directions, or single-feature thresholds.

In [ ]:
plot_confusion_matrices(test_predictions)

In [ ]:
plot_score_curves(test_predictions)